In [55]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import numpy as np
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.fit_params.weibull_fit_params import WeibullFitParams

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [56]:
power_law_db = pl.read_parquet(r"MLE_random_sample_fit_data\PowerLaw")
weibull_db = pl.read_parquet(r"MLE_random_sample_fit_data\Weibull").filter(pl.col("k_err").is_not_nan())
log_normal_db = pl.read_parquet(r"MLE_random_sample_fit_data\LogNormal").filter(pl.col("mu_err").is_not_nan())

In [ ]:
power_law_db

seed,aic,bic,numb_alphas,s_max_fitting_alpha,s_min_fitting_alpha,q,g_mu,g_std,q_err,g_mu_err,g_std_err,LAD_min,J_min,min_alpha_to_consider
i32,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64
91578,2.3019e6,2.3020e6,167084,34166.764882,23.497827,1.083324,0.418283,0.078257,NaN,NaN,NaN,0,0.7,117.303041
5305,2.3014e6,2.3015e6,167084,34166.764882,23.497827,1.033252,0.420658,0.079522,NaN,NaN,NaN,0,0.7,117.303041
96768,2.3025e6,2.3026e6,167084,34166.764882,23.497827,0.998937,0.422747,0.082179,NaN,NaN,NaN,0,0.7,117.303041
60831,2.3006e6,2.3007e6,167084,34166.764882,22.776133,0.9911,0.419741,0.081097,NaN,NaN,NaN,0,0.7,117.303041
62890,2.3009e6,2.3010e6,167084,34166.764882,22.776133,1.016585,0.431058,0.079371,NaN,NaN,NaN,0,0.7,117.303041
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
76869,478492.507099,478542.110464,28774,20845.303305,22.776133,1.191721,0.374086,0.048244,NaN,NaN,NaN,0,0.7,769.276497
2494,479372.521059,479422.124424,28774,20845.303305,23.497827,1.179511,0.377633,0.048791,NaN,NaN,NaN,0,0.7,769.276497
56509,478883.005392,478932.608757,28774,34166.764882,22.776133,1.170713,0.381061,0.048204,NaN,NaN,NaN,0,0.7,769.276497


In [ ]:
columns_to_select = ["aic", "numb_alphas"]

aic_db = pl.concat([
    power_law_db.select(columns_to_select).with_columns(pl.lit("PL").alias("fit_type")),
    weibull_db.select(columns_to_select).with_columns(pl.lit("W").alias("fit_type")),
    log_normal_db.select(columns_to_select).with_columns(pl.lit("LN").alias("fit_type"))
]).filter(pl.col("numb_alphas") == 412506)

aic_db

aic,numb_alphas,fit_type
f64,i64,str
2.3019e6,167084,"""PL"""
2.3014e6,167084,"""PL"""
2.3025e6,167084,"""PL"""
2.3006e6,167084,"""PL"""
2.3009e6,167084,"""PL"""
…,…,…
605739.792867,37294,"""LN"""
605516.04061,37294,"""LN"""
605339.267819,37294,"""LN"""


In [61]:
aic_db = aic_db.join(
        aic_db.group_by("numb_alphas").agg(pl.col("aic").min().alias("min_aic_for_numb_alphas")),
        on = "numb_alphas",
    ).with_columns(
        (pl.col("aic") - pl.col("min_aic_for_numb_alphas")).alias("delta_aic")
    )

aic_db

aic,numb_alphas,fit_type,min_aic_for_numb_alphas,delta_aic
f64,i64,str,f64,f64
2.3019e6,167084,"""PL""",2.2991e6,2778.623313
2.3014e6,167084,"""PL""",2.2991e6,2314.733962
2.3025e6,167084,"""PL""",2.2991e6,3419.91076
2.3006e6,167084,"""PL""",2.2991e6,1528.066507
2.3009e6,167084,"""PL""",2.2991e6,1819.558131
…,…,…,…,…
605739.792867,37294,"""LN""",604936.505642,803.287225
605516.04061,37294,"""LN""",604936.505642,579.534968
605339.267819,37294,"""LN""",604936.505642,402.762177


In [62]:
aic_db.group_by("fit_type").agg(
    pl.col("delta_aic").mean().alias("rel_mean_aic"),
    pl.col("delta_aic").std().alias("std_aic"),
    pl.col("delta_aic").min().alias("rel_min_aic"),
    pl.col("delta_aic").max().alias("rel_max_aic"),
    pl.len()
)

fit_type,rel_mean_aic,std_aic,rel_min_aic,rel_max_aic,len
str,f64,f64,f64,f64,u32
"""LN""",818.207273,1381.270243,0.0,8240.70638,299
"""PL""",936.015949,1686.218032,0.0,12405.984788,360
"""W""",34.967326,75.582889,0.0,264.44853,12


In [63]:
import polars as pl

types = aic_db["fit_type"].unique().sort().to_list()

samples = {
    t: aic_db.filter(pl.col("fit_type") == t)["delta_aic"].to_numpy()
    for t in types
}

for t in types:
    others = [o for o in types if o != t]

    a = samples[t]
    b = samples[others[0]]
    c = samples[others[1]]

    p = (
        (a[:, None, None] < b[None, :, None]) &
        (a[:, None, None] < c[None, None, :])
    ).mean()

    print(f"P(AIC {t} < ({others[0]}, {others[1]})) = {100*p:.3f}%")

P(AIC LN < (PL, W)) = 10.742%
P(AIC PL < (LN, W)) = 12.240%
P(AIC W < (LN, PL)) = 73.596%
